In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
REPO_URL = 'https://github.com/Dweeb1578/voicemos-2026.git'
REPO_DIR = '/content/voicemos-2026'
CKPT_DIR = '/content/drive/MyDrive/voicemos2026/checkpoints'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)


In [ ]:
!python data/download.py --output data/datasets --datasets bvcc tmhint audiomos25t3

In [ ]:
!python data/build_manifests.py --data_dir data/datasets --output_dir data/manifests

In [ ]:
# Pre-extract frozen Whisper encoder outputs (one-time, ~1-2h on T4).
# Saves ~47 GB to data/encoder_cache/. Skip if cache already exists.
import os
if not os.path.exists('data/encoder_cache') or len(os.listdir('data/encoder_cache')) < 100:
    !python -m data.cache_features \
        --manifests data/manifests/pretrain_train.csv data/manifests/pretrain_dev.csv \
        --cache_dir data/encoder_cache \
        --whisper_model openai/whisper-medium \
        --batch_size 32
else:
    print(f"Cache exists ({len(os.listdir('data/encoder_cache'))} files), skipping extraction.")

In [ ]:
!python src/train.py --config configs/pretrain.yaml

In [ ]:
import shutil, os
CKPT_DIR = '/content/drive/MyDrive/voicemos2026/checkpoints'
shutil.copy('checkpoints/pretrain/best.pt', f'{CKPT_DIR}/pretrain_best.pt')
print('Checkpoint saved to Drive.')


In [ ]:
import torch
from torch.utils.data import DataLoader
from src.model import WhisperMOSNet
from src.dataset import MOSDataset
from src.evaluate import compute_metrics

device = torch.device('cuda')
model = WhisperMOSNet(whisper_model='openai/whisper-medium', proj_dim=256).to(device)
ckpt = torch.load('checkpoints/pretrain/best.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.train(False)

ds = MOSDataset('data/manifests/pretrain_dev.csv', whisper_model='openai/whisper-medium')
loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)

acr_preds, acr_targets = [], []
with torch.no_grad():
    for batch in loader:
        acr, _ = model(batch['input_features'].to(device), batch['waveform'].to(device))
        acr_preds.extend(acr.cpu().tolist())
        acr_targets.extend(batch['acr'].tolist())

m = compute_metrics(acr_preds, acr_targets)
print(f"Dev ACR  SRCC={m['srcc']:.4f}  LCC={m['lcc']:.4f}  MSE={m['mse']:.4f}")
beat = 'BEAT' if m['srcc'] > 0.780 else 'not yet'
print(f'Baseline (MOSA-Net+): SRCC=0.780 -- {beat}')
